# Repeatome notebook (skeleton)

This notebook defines a lightweight `Repeatome` class to aggregate repeat annotations across genomes and run basic analyses.
Fill in the domain-specific details (expected columns, exact metrics, plotting style) where marked with `TODO`.


In [ ]:
# Core imports
from __future__ import annotations

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

# Optional: phylogeny parsing (pick one)
from Bio import Phylo
from io import StringIO


In [ ]:
class Repeatome:
    """A small container class for multi-genome repeat annotation analyses.

    Attributes
    ----------
    gff : pd.DataFrame
        Aggregated annotation table (GFF-derived CSV) across genomes.
    assembly_size : pd.DataFrame
        Assembly sizes across genomes (must include a genome_id column).
    landscape : pd.DataFrame
        Landscape / divergence table across genomes (must include a genome_id column).
    tree : object
        Parsed phylogenetic tree (e.g., Bio.Phylo tree).
    """

    def __init__(self):
        self.gff = pd.DataFrame()
        self.assembly_size = pd.DataFrame()
        self.landscape = pd.DataFrame()
        self.tree = None

        # results
        self.correlation = None
        self.plot_correlation = None

        self.overview = None
        self.plot_overview = None

        self.abundance = None
        self.plot_abundance = None

        self.plot_landscape = None

    # -------------------------
    # Loading / adding datasets
    # -------------------------
    def add_gff(self, name: str, file: str, **read_csv_kwargs):
        """Load a GFF-derived CSV into a DataFrame and append it to self.gff."""
        df = pd.read_csv(file, **read_csv_kwargs)
        df["genome_id"] = name
        self.gff = pd.concat([self.gff, df], ignore_index=True)
        return df

    def add_landscape(self, name: str, file: str, **read_csv_kwargs):
        """Load a landscape CSV into a DataFrame and append it to self.landscape."""
        df = pd.read_csv(file, **read_csv_kwargs)
        df["genome_id"] = name
        self.landscape = pd.concat([self.landscape, df], ignore_index=True)
        return df

    def add_size(self, name: str, file: str, **read_csv_kwargs):
        """Load an assembly size file (CSV/TSV) and append it to self.assembly_size."""
        df = pd.read_csv(file, **read_csv_kwargs)
        df["genome_id"] = name
        self.assembly_size = pd.concat([self.assembly_size, df], ignore_index=True)
        return df

    def add_phylogeny(self, newick_file: str):
        """Parse a Newick file and store it in self.tree."""
        self.tree = Phylo.read(newick_file, "newick")
        return self.tree

    # -------------------------
    # Filtering
    # -------------------------
    def filter_in_te(self, df: pd.DataFrame | None = None):
        """Keep only rows where the 3rd field equals 'Transposable_elements'.

        Notes
        -----
        If a column named 'type' exists, it is used.
        Otherwise, column index 2 is used (third column).
        """
        if df is None:
            df = self.gff

        if "type" in df.columns:
            out = df[df["type"] == "Transposable_elements"].copy()
        else:
            out = df[df.iloc[:, 2] == "Transposable_elements"].copy()

        return out

    # -------------------------
    # Analyses + plots
    # -------------------------
    def calculate_correlation(self, df: pd.DataFrame | None = None):
        """Compute a correlation table from the GFF table.

        TODO
        ----
        Define the exact feature(s) to correlate (counts, bp, % genome, per TE class, etc.).
        As a placeholder, this computes correlations on numeric columns grouped by genome_id.
        """
        if df is None:
            df = self.gff

        # Placeholder: genome-level means of numeric columns
        num = df.select_dtypes(include=["number"])
        if num.shape[1] == 0:
            self.correlation = pd.DataFrame()
            return self.correlation

        agg = df.groupby("genome_id")[num.columns].mean()
        self.correlation = agg.corr()
        return self.correlation

    def plot_correlation_matrix(self, corr: pd.DataFrame | None = None):
        """Plot a correlation heatmap (simple matplotlib)."""
        if corr is None:
            corr = self.correlation

        fig, ax = plt.subplots(figsize=(6, 5))
        im = ax.imshow(corr.values, aspect="auto")
        ax.set_xticks(range(len(corr.columns)))
        ax.set_yticks(range(len(corr.index)))
        ax.set_xticklabels(corr.columns, rotation=90)
        ax.set_yticklabels(corr.index)
        fig.colorbar(im, ax=ax, shrink=0.8)
        ax.set_title("Correlation matrix")
        fig.tight_layout()

        self.plot_correlation = fig
        return fig

    def calculate_overview(self, df: pd.DataFrame | None = None):
        """Compute a lightweight overview summary.

        TODO
        ----
        Replace with the overview metrics you need (e.g., TE bp, TE %, class breakdown, etc.).
        """
        if df is None:
            df = self.gff

        # Placeholder: row counts per genome
        self.overview = df.groupby("genome_id").size().rename("n_rows").reset_index()
        return self.overview

    def plot_overview_summary(self, overview: pd.DataFrame | None = None):
        """Plot overview summary (placeholder)."""
        if overview is None:
            overview = self.overview

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(overview["genome_id"], overview.iloc[:, 1])
        ax.set_ylabel(overview.columns[1])
        ax.set_xlabel("genome_id")
        ax.set_title("Overview")
        ax.tick_params(axis='x', rotation=90)
        fig.tight_layout()

        self.plot_overview = fig
        return fig

    def calculate_abundance(self, df: pd.DataFrame | None = None):
        """Compute abundance metrics from GFF.

        TODO
        ----
        Decide whether abundance is counts, summed lengths, normalized by assembly size, etc.
        """
        if df is None:
            df = self.gff

        # Placeholder: counts per genome
        self.abundance = df.groupby("genome_id").size().rename("count").reset_index()
        return self.abundance

    def draw_abundance(self, abundance: pd.DataFrame | None = None):
        """Plot abundance (placeholder)."""
        if abundance is None:
            abundance = self.abundance

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(abundance["genome_id"], abundance["count"])
        ax.set_ylabel("count")
        ax.set_xlabel("genome_id")
        ax.set_title("Abundance")
        ax.tick_params(axis='x', rotation=90)
        fig.tight_layout()

        self.plot_abundance = fig
        return fig

    def plot_landscape_from_table(self, landscape: pd.DataFrame | None = None):
        """Plot a landscape / divergence distribution (placeholder).

        TODO
        ----
        Define which columns encode divergence (e.g., 'kimura', 'divergence') and abundance.
        """
        if landscape is None:
            landscape = self.landscape

        # Placeholder: if there's a 'divergence' column, histogram it
        if "divergence" not in landscape.columns:
            self.plot_landscape = None
            return None

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(landscape["divergence"].dropna().values, bins=30)
        ax.set_xlabel("divergence")
        ax.set_ylabel("frequency")
        ax.set_title("Landscape (divergence)")
        fig.tight_layout()

        self.plot_landscape = fig
        return fig


In [ ]:
# Create a Repeatome object
repeatome = Repeatome()

# Example usage (fill paths and CSV formats)
# repeatome.add_gff("genomeA", "path/to/genomeA.gff.csv")
# repeatome.add_size("genomeA", "path/to/genomeA.size.tsv", sep="\t")
# repeatome.add_landscape("genomeA", "path/to/genomeA.landscape.csv")
# repeatome.add_phylogeny("path/to/tree.nwk")


In [ ]:
# Example analysis flow (placeholders)
# te_gff = repeatome.filter_in_te()

# corr = repeatome.calculate_correlation()
# repeatome.plot_correlation_matrix()

# overview = repeatome.calculate_overview()
# repeatome.plot_overview_summary()

# abundance = repeatome.calculate_abundance()
# repeatome.draw_abundance()

# repeatome.plot_landscape_from_table()
